# Campeonato Inteli Módulo 3 2026 - Notebook final (submissões v3 e v4)

Este notebook documenta o processo completo que gerou as duas submissões escolhidas
como finais na competição: `submission_v3.csv` e `submission_v4.csv`. As duas usam um
blend (média ponderada) de três modelos - regressão logística, Random Forest e
HistGradientBoosting - combinando features derivadas do Elo e da forma recente dos
times.

Regras seguidas:
- Apenas NumPy, Pandas e Scikit-Learn para modelagem.
- Nenhum dado externo: apenas `train.csv` e `test.csv` fornecidos pela competição.
- Cada previsão usa apenas as colunas da própria linha do `test.csv` (mais o que foi
  aprendido a partir do `train.csv`). Nenhuma comparação entre linhas diferentes do
  teste é usada para inferir resultados.
- Validação sempre respeitando a ordem temporal (treino no passado, validação no
  "futuro"), nunca embaralhando as partidas.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import log_loss

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print(train.shape, test.shape)
train.head()

(3419, 26) (1725, 22)


,Id,Season,Date,Home,Away,elo_home,elo_away,elo_diff,form_pts_home,form_pts_away,...,rest_days_home,rest_days_away,season_ppg_home,season_ppg_away,round_n,h2h_home_pts,Res,odd_casa,odd_empate,odd_visitante
0,0,2013,2013-05-25,Vasco,Portuguesa,1506.636452,1486.580590,20.055862,1.6,1.0,...,174.0,174.0,NaN,NaN,1,3.0,H,1.86,3.44,4.06
1,1,2013,2013-05-25,Vitoria,Internacional,1500.000000,1490.971962,9.028038,NaN,0.2,...,NaN,174.0,NaN,NaN,1,NaN,D,2.72,3.17,2.52
2,2,2013,2013-05-26,Corinthians,Botafogo RJ,1527.354763,1506.560479,20.794285,2.0,1.0,...,175.0,176.0,NaN,NaN,1,0.0,D,1.82,3.43,4.21
3,3,2013,2013-05-26,Criciuma,Bahia,1500.000000,1498.594997,1.405003,NaN,2.0,...,NaN,175.0,NaN,NaN,1,NaN,H,1.72,3.50,4.70
4,4,2013,2013-05-26,Gremio,Nautico,1561.732962,1489.510755,72.222206,2.2,1.4,...,175.0,175.0,NaN,NaN,1,3.0,H,1.38,4.36,8.19


## 1. Features

Partimos das 17 colunas originais (Elo, forma recente, descanso, histórico de
confronto direto) e acrescentamos diferenças diretas entre mandante e visitante, além
de algumas transformações simples (`elo_diff` ao quadrado, `h2h_home_pts`
centralizado, indicador de início de temporada, e uma probabilidade estilo Elo). Todas
calculadas linha a linha - nenhuma usa outra partida do teste. As colunas de odds
(`odd_casa`, `odd_empate`, `odd_visitante`) existem só no treino e por isso não
entram como feature.

In [2]:
BASE_FEATURES = [
    'elo_home', 'elo_away', 'elo_diff',
    'form_pts_home', 'form_pts_away',
    'form_gf_home', 'form_ga_home', 'form_gf_away', 'form_ga_away',
    'home_form_as_host', 'away_form_as_visitor',
    'rest_days_home', 'rest_days_away',
    'season_ppg_home', 'season_ppg_away',
    'round_n', 'h2h_home_pts',
]

def engineer(df):
    df = df.copy()
    df['form_pts_diff'] = df['form_pts_home'] - df['form_pts_away']
    df['form_gf_diff'] = df['form_gf_home'] - df['form_gf_away']
    df['form_ga_diff'] = df['form_ga_home'] - df['form_ga_away']
    df['season_ppg_diff'] = df['season_ppg_home'] - df['season_ppg_away']
    df['rest_diff'] = df['rest_days_home'] - df['rest_days_away']
    df['home_net_form'] = df['form_gf_home'] - df['form_ga_home']
    df['away_net_form'] = df['form_gf_away'] - df['form_ga_away']
    df['net_form_diff'] = df['home_net_form'] - df['away_net_form']
    df['host_visitor_form_diff'] = df['home_form_as_host'] - df['away_form_as_visitor']
    df['elo_diff_sq'] = np.sign(df['elo_diff']) * (df['elo_diff'] ** 2) / 1000.0
    df['h2h_centered'] = df['h2h_home_pts'] - 1.5
    df['is_early_season'] = (df['round_n'] <= 3).astype(float)
    df['elo_logistic'] = 1.0 / (1.0 + 10 ** (-df['elo_diff'] / 400.0))
    return df

train_e = engineer(train)
test_e = engineer(test)

FEATURES_V3 = BASE_FEATURES + [
    'form_pts_diff', 'form_gf_diff', 'form_ga_diff', 'season_ppg_diff',
    'rest_diff', 'home_net_form', 'away_net_form',
    'net_form_diff', 'host_visitor_form_diff', 'elo_diff_sq', 'h2h_centered', 'is_early_season',
]
FEATURES_V4 = FEATURES_V3 + ['elo_logistic']  # v4 = v3 + probabilidade estilo Elo

LABEL_ORDER = ['A', 'D', 'H']  # ordem alfabetica exigida no CSV de submissao

def make_pipeline(clf):
    return Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('clf', clf),
    ])

def ordered_proba(pipe, X):
    proba = pipe.predict_proba(X)
    classes = list(pipe.named_steps['clf'].classes_)
    return proba[:, [classes.index(c) for c in LABEL_ORDER]]

## 2. Validação temporal (walk-forward por temporada)

Treinamos com tudo antes da temporada `s` e avaliamos só na temporada `s`, para
`s` em `[2018, 2019, 2020, 2021]`. Isso imita a situação real do teste (treino no
passado, previsão no futuro).

In [3]:
VAL_SEASONS = [2018, 2019, 2020, 2021]

def eval_blend(features, weights, l2_hgb=2.0, C_lr=0.03, rf_depth=4, rf_leaf=20):
    losses = []
    for s in VAL_SEASONS:
        tr_mask = train_e['Season'] < s
        val_mask = train_e['Season'] == s
        X_tr, y_tr = train_e.loc[tr_mask, features], train_e.loc[tr_mask, 'Res']
        X_val, y_val = train_e.loc[val_mask, features], train_e.loc[val_mask, 'Res']

        lr = make_pipeline(LogisticRegression(max_iter=3000, C=C_lr)); lr.fit(X_tr, y_tr)
        rf = make_pipeline(RandomForestClassifier(n_estimators=500, max_depth=rf_depth, min_samples_leaf=rf_leaf, random_state=42)); rf.fit(X_tr, y_tr)
        hgb = make_pipeline(HistGradientBoostingClassifier(max_depth=2, max_iter=100, learning_rate=0.03, l2_regularization=l2_hgb, random_state=42)); hgb.fit(X_tr, y_tr)

        blend = weights[0] * ordered_proba(lr, X_val) + weights[1] * ordered_proba(rf, X_val) + weights[2] * ordered_proba(hgb, X_val)
        losses.append(log_loss(y_val, blend, labels=LABEL_ORDER))
    return np.mean(losses)

ll_v3 = eval_blend(FEATURES_V3, weights=(0.5, 0.2, 0.3))
ll_v4 = eval_blend(FEATURES_V4, weights=(0.5, 0.25, 0.25))
print('v3 walk-forward logloss:', ll_v3)
print('v4 walk-forward logloss:', ll_v4)

C:\Users\Inteli\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


C:\Users\Inteli\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


C:\Users\Inteli\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


C:\Users\Inteli\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


C:\Users\Inteli\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


C:\Users\Inteli\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


C:\Users\Inteli\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


C:\Users\Inteli\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


v3 walk-forward logloss: 1.0157216702937677
v4 walk-forward logloss: 1.015693935102999


Os pesos do blend (`0.5 / 0.2 / 0.3` na v3, `0.5 / 0.25 / 0.25` na v4) e os
hiperparâmetros de cada modelo (`C=0.03` na regressão logística, `max_depth=4,
min_samples_leaf=20` na Random Forest, `max_depth=2, learning_rate=0.03,
l2_regularization=2.0` no HistGradientBoosting) foram escolhidos por meio de uma busca
em grade nessa mesma validação walk-forward, testando várias combinações e ficando com
a de menor LogLoss médio. Os dois blends ficam bem abaixo dos benchmarks da régua da
competição (1.0594 sem modelo, 1.047 do notebook de partida).

## 3. Submissão v3

Blend `0.5 * logreg + 0.2 * RF + 0.3 * HGB`, treinado em todo o `train.csv` e aplicado
ao `test.csv`.

In [4]:
X3, y3 = train_e[FEATURES_V3], train_e['Res']
Xt3 = test_e[FEATURES_V3]

lr_v3 = make_pipeline(LogisticRegression(max_iter=3000, C=0.03))
rf_v3 = make_pipeline(RandomForestClassifier(n_estimators=500, max_depth=4, min_samples_leaf=20, random_state=42))
hgb_v3 = make_pipeline(HistGradientBoostingClassifier(max_depth=2, max_iter=100, learning_rate=0.03, l2_regularization=2.0, random_state=42))
lr_v3.fit(X3, y3); rf_v3.fit(X3, y3); hgb_v3.fit(X3, y3)

p_v3 = 0.5 * ordered_proba(lr_v3, Xt3) + 0.2 * ordered_proba(rf_v3, Xt3) + 0.3 * ordered_proba(hgb_v3, Xt3)

submission_v3 = pd.DataFrame({'Id': test_e['Id'], 'A': p_v3[:, 0], 'D': p_v3[:, 1], 'H': p_v3[:, 2]})
assert np.allclose(submission_v3[['A', 'D', 'H']].sum(axis=1), 1.0)
submission_v3.to_csv('submission_v3.csv', index=False)
submission_v3.head()

C:\Users\Inteli\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


,Id,A,D,H
0,3419,0.189891,0.305313,0.504797
1,3420,0.262118,0.332608,0.405274
2,3421,0.168851,0.265574,0.565575
3,3422,0.280472,0.213412,0.506116
4,3423,0.131858,0.215425,0.652717


## 4. Submissão v4

Mesma ideia, acrescentando a feature `elo_logistic` e ajustando os pesos do blend:
`0.5 * logreg + 0.25 * RF + 0.25 * HGB`.

In [5]:
X4, y4 = train_e[FEATURES_V4], train_e['Res']
Xt4 = test_e[FEATURES_V4]

lr_v4 = make_pipeline(LogisticRegression(max_iter=3000, C=0.03))
rf_v4 = make_pipeline(RandomForestClassifier(n_estimators=500, max_depth=4, min_samples_leaf=20, random_state=42))
hgb_v4 = make_pipeline(HistGradientBoostingClassifier(max_depth=2, max_iter=100, learning_rate=0.03, l2_regularization=2.0, random_state=42))
lr_v4.fit(X4, y4); rf_v4.fit(X4, y4); hgb_v4.fit(X4, y4)

p_v4 = 0.5 * ordered_proba(lr_v4, Xt4) + 0.25 * ordered_proba(rf_v4, Xt4) + 0.25 * ordered_proba(hgb_v4, Xt4)

submission_v4 = pd.DataFrame({'Id': test_e['Id'], 'A': p_v4[:, 0], 'D': p_v4[:, 1], 'H': p_v4[:, 2]})
assert np.allclose(submission_v4[['A', 'D', 'H']].sum(axis=1), 1.0)
submission_v4.to_csv('submission_v4.csv', index=False)
submission_v4.head()

C:\Users\Inteli\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


,Id,A,D,H
0,3419,0.191479,0.303993,0.504527
1,3420,0.266318,0.326082,0.407600
2,3421,0.168343,0.266283,0.565375
3,3422,0.281713,0.215461,0.502826
4,3423,0.132024,0.213655,0.654321


## 5. Verificação anti-vazamento

Todas as features (`FEATURES_V3`, `FEATURES_V4`) são calculadas linha a linha, usando
só as colunas da própria partida. Nenhum modelo (`lr_v3`, `rf_v3`, `hgb_v3`, `lr_v4`,
`rf_v4`, `hgb_v4`) é ajustado com dados de `test.csv` - todos usam apenas
`train.csv`. Nenhuma coluna de estado (`elo`, `form`, `season_ppg`) é comparada entre
diferentes linhas do teste. A ordem original das linhas (por data) é preservada
durante toda a validação - nunca usamos `shuffle`, e todas as divisões treino/validação
respeitam a ordem temporal (treino sempre estritamente anterior à temporada de
validação).